# Notebook 04: Model Development

Train regression models to predict Expected Post Reach.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("../data/processed/model_data.csv")
X = df.drop("Reach", axis=1)
y = df["Reach"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Preprocessing Pipeline

In [3]:
numeric_features = ["day_of_week", "month", "hashtag_count", "is_weekend"]
categorical_features = ["Post Type", "Campaign Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ])

## Model Training and Comparison

In [4]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

results_df = pd.DataFrame(results)
display(results_df)

,Model,RMSE,MAE,R2
0,Linear Regression,11280.608015,9587.783157,-0.000520
1,Decision Tree,12032.870872,10054.217295,-0.138412
2,Random Forest,12024.039426,10050.086490,-0.136741
3,Gradient Boosting,11288.436675,9593.350498,-0.001909


## Select and Save Final Model
We will choose Random Forest (or the best performing model).

In [5]:
best_model_name = results_df.loc[results_df["RMSE"].idxmin()]["Model"]
print(f"Best Model: {best_model_name}")

final_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", models[best_model_name])])
final_pipeline.fit(X, y) # Train on full data for deployment

os.makedirs("../models", exist_ok=True)
joblib.dump(final_pipeline, "../models/final_model.joblib")
print("Model saved to ../models/final_model.joblib")

Best Model: Linear Regression
Model saved to ../models/final_model.joblib
